# Reevaluación FOM-5 v2 — sin volver a ejecutar Qwen

Este notebook **NO vuelve a generar ninguna formulación**. Reutiliza `resultados_fom5.csv` y corrige únicamente la etapa de evaluación.

Motivo: el judge local Qwen2.5-7B fue demasiado permisivo. Dos ejemplos detectados manualmente:

- `opt_02` recibió 5/5, aunque reemplaza \(x_C\) por la variable de horas extra en las restricciones de recursos y omite el costo de horas extra del objetivo.
- `opt_10` recibió 5/5, aunque no formula un modelo matemático y además interpreta mal varias condiciones.

FOM-5 v2 cambia el procedimiento:

1. evalúa cada requisito semántico **uno por uno**;
2. obliga al judge a justificar cada estado;
3. calcula parte de los puntajes mediante código, no dejando los cinco números totalmente a criterio del LLM;
4. aplica topes cuando no existe una formulación matemática real;
5. ejecuta pruebas de calibración antes de congelar el judge.

Los gold/reference siguen usándose **solo para evaluar**, nunca para generar, seleccionar o reparar respuestas.


In [ ]:
# Colab ya incluye pandas y requests. No actualizamos pandas/requests.
!pip -q install tqdm


In [ ]:
import os, io, re, json, time, gzip, base64, getpass
import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm
from google.colab import files

# Golds auditados embebidos.
GOLD = json.loads(gzip.decompress(base64.b64decode('...')).decode("utf-8"))
GOLD_LOOKUP = {x["id"]: x for x in GOLD}

# Sube el CSV producido por el notebook anterior.
uploaded = files.upload()
csv_name = next((n for n in uploaded if n.lower().endswith(".csv")), None)
if csv_name is None:
    raise ValueError("Debes subir resultados_fom5.csv")

old_df = pd.read_csv(io.BytesIO(uploaded[csv_name]))
print("Filas cargadas:", len(old_df))
display(old_df[["id","variant","score_fom5"]].head())


## Judge externo fijo

Se usa un judge más capaz únicamente para **evaluación offline**. No forma parte del sistema que se demuestra en la entrega y no interviene en la generación de Qwen2.5-1.5B.

Por defecto:
`qwen/qwen3-30b-a3b-instruct-2507`

El modelo evaluado continúa siendo `Qwen/Qwen2.5-1.5B-Instruct`.


In [ ]:
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY") or getpass.getpass("OpenRouter API key: ")
JUDGE_MODEL = "qwen/qwen3-30b-a3b-instruct-2507"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

print("Judge:", JUDGE_MODEL)


In [ ]:
SYSTEM_JUDGE = r"""
Eres un evaluador estricto de formulaciones de Investigación de Operaciones.
La tarea del candidato es FORMULAR un modelo PL/PLEM desde lenguaje natural, NO resolver numéricamente la instancia.

Reglas obligatorias:
1. Evalúa la respuesta tal como está escrita. No completes mentalmente ecuaciones faltantes.
2. Una explicación verbal, un algoritmo, código para resolver o una solución numérica NO sustituyen una formulación matemática.
3. Una restricción esencial omitida, invertida o aplicada a la variable equivocada cuenta como missing/wrong.
4. Si una ecuación contradice otra, penaliza algebra_validity y constraints_logic.
5. Si una variable decisional esencial no existe (por ejemplo apertura en localización), decision_domains no puede superar 2.
6. Si el sentido del objetivo es correcto pero falta un término esencial de costo/beneficio, objective no puede superar 2.
7. Si no existe función objetivo matemática, objective <= 1.
8. Si la respuesta no contiene una formulación matemática identificable, has_formulation=false.
9. Acepta formulaciones matemáticamente equivalentes al gold; no exijas los mismos símbolos ni la misma técnica.
10. No premies prosa extensa. No castigues una notación diferente si es coherente y equivalente.

Escala de dimensiones 0--5:
0 = ausente o totalmente inválido
1 = falla fundamental
2 = estructura reconocible con error(es) esenciales
3 = mayormente correcta, pero falta o falla una pieza material
4 = correcta salvo un detalle menor
5 = completa y matemáticamente correcta/equivalente

Debes revisar TODOS los requisitos indicados.
"""

def judge_prompt(item, candidate):
    req_lines = "\n".join(
        f'{r["id"]} [{r["kind"]}] {"CORE" if r["critical"] else ""}: {r["text"]}'
        for r in item["requirements"]
    )
    return f"""
ENUNCIADO
=========
{item["problem"]}

REQUISITOS AUDITADOS
====================
{req_lines}

FORMULACIÓN GOLD
================
{item["reference_model"]}

RESPUESTA CANDIDATA
===================
{candidate}

Devuelve SOLO JSON válido con esta forma exacta:
{{
  "has_formulation": true,
  "requirements": [
    {{
      "id": "R1",
      "status": "met|partial|missing|wrong",
      "reason": "explicación concreta comparando candidato y requisito"
    }}
  ],
  "raw_dimensions": {{
    "decision_domains": {{"score": 0, "reason": "..."}},
    "objective": {{"score": 0, "reason": "..."}},
    "constraints_logic": {{"score": 0, "reason": "..."}},
    "algebra_validity": {{"score": 0, "reason": "..."}},
    "generalization": {{"score": 0, "reason": "..."}}
  }},
  "fatal_errors": ["..."],
  "summary": "..."
}}

Incluye exactamente un resultado por cada requisito R1...Rn.
No agregues markdown ni texto fuera del JSON.
"""

def call_openrouter(prompt, max_retries=4):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": JUDGE_MODEL,
        "temperature": 0,
        "messages": [
            {"role":"system","content":SYSTEM_JUDGE},
            {"role":"user","content":prompt},
        ],
        "response_format": {"type":"json_object"},
    }
    last_err = None
    for k in range(max_retries):
        try:
            r = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=180)
            if r.status_code >= 400:
                # Fallback if a provider rejects response_format.
                if "response_format" in payload:
                    payload.pop("response_format", None)
                    continue
                r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"]
        except Exception as e:
            last_err = e
            time.sleep(2**k)
    raise RuntimeError(f"OpenRouter falló: {last_err}")

def extract_json(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a,b = text.find("{"), text.rfind("}")
    if a < 0 or b < a:
        raise ValueError("No se encontró JSON.")
    return json.loads(text[a:b+1])


## Agregación determinista

El judge propone scores brutos, pero FOM-5 v2 aplica topes verificables:

- el requisito de objetivo limita `objective`;
- los requisitos de restricciones limitan `constraints_logic`;
- un requisito explícito de dominios limita `decision_domains`;
- si `has_formulation=false`, el total no puede exceder 1.5/5;
- `missing` y `wrong` valen 0, `partial` vale 0.5 y `met` vale 1.

Así un judge ya no puede dar 5/5 a una respuesta que reconoce como incompleta requisito por requisito.


In [ ]:
STATUS_VALUE = {"met":1.0, "partial":0.5, "missing":0.0, "wrong":0.0}

def safe_score(x):
    x = float(x)
    return max(0.0, min(5.0, x))

def status_map(parsed):
    out = {}
    for r in parsed.get("requirements", []):
        sid = str(r.get("id",""))
        st = str(r.get("status","")).lower().strip()
        out[sid] = STATUS_VALUE.get(st, 0.0)
    return out

def aggregate_v2(item, parsed):
    sm = status_map(parsed)
    raw = parsed["raw_dimensions"]

    def rawdim(name):
        return safe_score(raw[name]["score"])

    req_by_kind = {"objective":[], "constraint":[], "domain":[]}
    for r in item["requirements"]:
        req_by_kind[r["kind"]].append(sm.get(r["id"], 0.0))

    obj_cov = np.mean(req_by_kind["objective"]) if req_by_kind["objective"] else 1.0
    con_cov = np.mean(req_by_kind["constraint"]) if req_by_kind["constraint"] else 1.0
    dom_cov = np.mean(req_by_kind["domain"]) if req_by_kind["domain"] else 1.0

    decision = rawdim("decision_domains")
    objective = min(rawdim("objective"), 5.0*obj_cov)
    constraints = min(rawdim("constraints_logic"), 5.0*con_cov)
    algebra = rawdim("algebra_validity")
    generalization = rawdim("generalization")

    if req_by_kind["domain"]:
        decision = min(decision, 5.0*dom_cov)

    score = (
        0.15*decision +
        0.20*objective +
        0.35*constraints +
        0.15*algebra +
        0.15*generalization
    )

    if not bool(parsed.get("has_formulation", True)):
        score = min(score, 1.5)
        algebra = min(algebra, 1.0)
        generalization = min(generalization, 1.0)
        score = min(score, (
            0.15*decision + 0.20*objective + 0.35*constraints +
            0.15*algebra + 0.15*generalization
        ))

    all_cov = np.mean(list(sm.values())) if sm else np.nan

    return {
        "score_fom5_v2": round(float(score), 3),
        "decision_domains_v2": decision,
        "objective_v2": objective,
        "constraints_logic_v2": constraints,
        "algebra_validity_v2": algebra,
        "generalization_v2": generalization,
        "requirement_coverage_v2": float(all_cov) if not np.isnan(all_cov) else np.nan,
        "objective_coverage": float(obj_cov),
        "constraint_coverage": float(con_cov),
        "domain_coverage": float(dom_cov),
    }


In [ ]:
# Ejecutar judge sobre las 32 respuestas ya guardadas.
rows = []

for _, rec in tqdm(old_df.iterrows(), total=len(old_df), desc="Reevaluando FOM-5 v2"):
    item = GOLD_LOOKUP[rec["id"]]
    parsed = None
    raw_text = None
    error = None

    for attempt in range(2):
        try:
            raw_text = call_openrouter(judge_prompt(item, rec["candidate"]))
            parsed = extract_json(raw_text)

            expected_ids = {r["id"] for r in item["requirements"]}
            got_ids = {str(r.get("id","")) for r in parsed.get("requirements",[])}
            if expected_ids != got_ids:
                raise ValueError(f"Requisitos incompletos. Esperados={expected_ids}; recibidos={got_ids}")
            agg = aggregate_v2(item, parsed)
            error = None
            break
        except Exception as e:
            error = str(e)
            parsed = None

    out = rec.to_dict()
    if parsed is None:
        out.update({"score_fom5_v2":np.nan, "judge_v2_error":error, "judge_v2_raw":raw_text})
    else:
        out.update(agg)
        out.update({
            "has_formulation_v2": bool(parsed.get("has_formulation",True)),
            "fatal_errors_v2": json.dumps(parsed.get("fatal_errors",[]), ensure_ascii=False),
            "judge_v2_summary": parsed.get("summary",""),
            "requirements_v2": json.dumps(parsed.get("requirements",[]), ensure_ascii=False),
            "judge_v2_raw": json.dumps(parsed, ensure_ascii=False),
            "judge_v2_error": None,
        })
    rows.append(out)

v2 = pd.DataFrame(rows)
v2.to_csv("/content/resultados_fom5_v2.csv", index=False, encoding="utf-8-sig")
v2.to_json("/content/resultados_fom5_v2.jsonl", orient="records", lines=True, force_ascii=False)

display(v2[[
    "id","variant","score_fom5","score_fom5_v2",
    "requirement_coverage","requirement_coverage_v2",
    "fatal_errors_v2"
]])


In [ ]:
# Calibración manual: estas NO son notas oficiales;
# son pruebas de sanidad para evitar repetir el fallo del judge anterior.
checks = [
    ("opt_02","baseline_direct_deterministic",2.8,
     "restricciones de recursos usan Y en lugar de x_C y falta costo de horas extra"),
    ("opt_10","baseline_direct_deterministic",1.5,
     "no entrega formulación y malinterpreta reglas"),
    ("opt_14","baseline_direct_deterministic",1.5,
     "resuelve en vez de formular; faltan balances/capacidades"),
    ("opt_16","baseline_direct_deterministic",1.8,
     "flujo multicommodity estructuralmente incorrecto"),
    ("diagnostic_oftalmo","entrega1_historical_sampled",2.5,
     "falta apertura, ponderación y hay restricciones erróneas"),
]

print("PRUEBAS DE CALIBRACIÓN")
all_ok = True
for iid, var, maximum, why in checks:
    r = v2[(v2.id==iid)&(v2.variant==var)]
    if r.empty:
        print("⚠️", iid, "no encontrado")
        all_ok=False
        continue
    s = float(r.iloc[0].score_fom5_v2)
    ok = s <= maximum
    all_ok &= ok
    print(("OK" if ok else "REVISAR"), f"{iid}: {s:.3f} <= {maximum}", "|", why)

if all_ok:
    print("\nCalibración superada. El judge puede congelarse para los experimentos.")
else:
    print("\nNo congeles todavía el judge: revisa los casos marcados REVISAR.")


In [ ]:
# Resumen baseline v2 (solo 30 problemas; excluye diagnóstico).
base = v2[
    (v2["split"]=="benchmark") &
    (v2["variant"]=="baseline_direct_deterministic")
].copy()

print("BASELINE FOM-5 v2")
print({
    "n":len(base),
    "mean":round(base.score_fom5_v2.mean(),3),
    "median":round(base.score_fom5_v2.median(),3),
    "std":round(base.score_fom5_v2.std(ddof=1),3),
    "min":round(base.score_fom5_v2.min(),3),
    "max":round(base.score_fom5_v2.max(),3),
    "requirement_coverage":round(base.requirement_coverage_v2.mean(),3),
})

print("\nCaso oftalmológico:")
display(v2[v2.id=="diagnostic_oftalmo"][[
    "variant","score_fom5","score_fom5_v2",
    "requirement_coverage_v2","fatal_errors_v2","judge_v2_summary"
]])

print("\nDistribución:")
display(base["score_fom5_v2"].describe().to_frame("FOM-5 v2"))

files.download("/content/resultados_fom5_v2.csv")


## Cuándo congelar el baseline

No cambies nuevamente el evaluador después de comenzar las variantes de mejora.

El procedimiento correcto será:

`baseline (ya generado) → FOM-5 v2 congelado → IR+compilador → mismo FOM-5 v2 → Best-of-N → mismo FOM-5 v2 → Repair → mismo FOM-5 v2`

Si la calibración falla, ajusta el judge **antes** de ejecutar IR. Una vez que pase la calibración, no se cambia ni prompt, ni modelo judge, ni agregación.
